## Example: Iterative Bayesian updating with a Gaussian mean model (RWMH posterior + PPC)

### Goal
We simulate 3 incoming batches of data. After each batch we:
1) produce an updated posterior object over the parameter vector $\mu \in \mathbb{R}^2$,  
2) generate a small posterior predictive forecast, and  
3) compute a posterior predictive check (PPC) p-value based on a chosen statistic.

This is meant to showcase the “distributions-in → distributions-out” workflow pattern.

In [3]:
import numpy as np

from probpipe import (
    SimpleLikelihood,
    RWMH, 
    IterativeForecaster,
    PosteriorPredictiveChecker,
    Gaussian
    )


### Probabilistic model

**Unknown parameter**
We treat the unknown parameter as a 2D mean vector:
$\mu \in \mathbb{R}^2$

**Prior**
$\mu \sim \mathcal{N}(m_0, \Sigma_0)$, 
$m_0 = \begin{bmatrix}0\\0\end{bmatrix}$, 
$\Sigma_0 = I_2.$
`prior` encodes $p(\mu) = \mathcal{N}(m_0,\Sigma_0)$.

In [4]:
prior = Gaussian(mean=np.array([0.0, 0.0]), cov=np.eye(2))


**Likelihood wrapper**

`SimpleLikelihood` defines a likelihood family where:

$x \mid \mu \sim \mathcal{N}(\mu, I_2)$,
i.e. the unknown parameter is plugged into the distribution’s `mean` field, while `cov` is fixed.

For a batch $D=\{x_1,\dots,x_n\}$,
$\log p(D\mid\mu)=\sum_{j=1}^n \log \mathcal{N}(x_j\mid \mu, I_2)$.

In [5]:
likelihood = SimpleLikelihood(dist_cls=Gaussian, params_name="mean", cov=np.eye(2))







**Posterior**
$p(\mu \mid D) \propto p(D \mid \mu)\,p(\mu)$.

Even though this model is conjugate (Gaussian prior + Gaussian likelihood), in this example we intentionally compute an approximate posterior using MCMC to demonstrate the ProbPipe workflow structure.



### `RWMH` 

Although the class is named `RWMH`, the current implementation does not run Metropolis–Hastings.
Instead it constructs a crude approximation:

$\hat\mu = \frac{1}{n}\sum_{j=1}^n x_j$,

$q(\mu\mid D) := \mathcal{N}(\hat\mu, I_2)$.


In [6]:
approx_post = RWMH(step_size=0.1)


`forecaster` orchestrates the update → posterior → forecast loop:

- `.forecast(n_samples=10)` calls `posterior.sample_predictive(n_samples)` which does:
  -  sample a single parameter draw $\mu^{(1)} \sim q(\mu\mid D)$,
  - generate $n\_{samples}$ synthetic observations

    $x^{*}_1,\dots,x^{*}_{n\_{samples}} \sim \mathcal{N}(\mu^{(1)}, I_2)$.

  - So the forecast is a batch of generated observations, conditional on one posterior draw of $\mu$.

- Each `.update(data=obs_data)` returns a `PosteriorDistribution` whose sampler is this Gaussian approximation (see below).

In [7]:
forecaster = IterativeForecaster(prior=prior, likelihood=likelihood, approx_post=approx_post)

### Posterior predictive forecast

Given a statistic function $T(\cdot)$, the checker computes:
- observed statistic: $T(D_{\text{obs}})$ using the stored batch `posterior.data`,
- simulated statistics: for $k=1,\dots,200$,
  draw one replicated observation set of size 1:
  
  $D^{rep}_k \sim p(\cdot \mid \mu^{(k)}), \quad \mu^{(k)} \sim q(\mu \mid D_{\text{obs}})$,
  then compute $T(D^{rep}_k)$,
- and returns the one-sided tail probability:
$p_{\text{PPC}} = \frac{1}{200}\sum_{k=1}^{200}\mathbf{1}\{T(D^{rep}_k) \ge T(D_{\text{obs}})\}$.

**Note:** In this implementation each replicate uses `n_samples=1`, while the observed statistic is computed on the full observed batch (e.g., 100 points). That is a toy PPC and can behave differently from a “matched sample size” PPC.

In [8]:
ppc = PosteriorPredictiveChecker(statistic=np.mean)

### Data generation in the loop

Each iteration simulates a new batch of size $n=100$ from:
$x_j \sim \mathcal{N}\left(\begin{bmatrix}5\\-3\end{bmatrix},\,4I_2\right)$.

Meanwhile the likelihood inside the model uses covariance $I_2$, so the model is intentionally mis-specified in noise scale (it assumes less variance than the data generator).

In [10]:
forecasts = []
ppc_pvalues = []

for i in range(3):
    obs_data = np.random.multivariate_normal(mean=np.array([5.0, -3.0]), cov=4*np.eye(2), size=100)
    posterior_dist = forecaster.update(data=obs_data)
    forecast = forecaster.forecast(n_samples=10)

    forecasts.append(forecast)
    ppc_pvalues.append(ppc.predictive_p_value(posterior=posterior_dist))

    print(f"Posterior distribution after dataset {i}: {posterior_dist}")
    print(f"[{i}] ppc p-value: {ppc_pvalues[-1]}")
    

Posterior distribution after dataset 0: Gaussian([ 5.17536394 -3.01589935],[[1. 0.]
 [0. 1.]])
[0] ppc p-value: 0.5
Posterior distribution after dataset 1: Gaussian([ 5.19687311 -2.77056293],[[1. 0.]
 [0. 1.]])
[1] ppc p-value: 0.53
Posterior distribution after dataset 2: Gaussian([ 4.72696242 -3.07230341],[[1. 0.]
 [0. 1.]])
[2] ppc p-value: 0.54
